# SECAI — Data Cleaning
Loads configuration from `config/secai_config.json`.
Each cell tackles one specific issue found during initial screening.

In [ ]:
import re
import json
import pandas as pd

with open("../config/secai_config.json", "r", encoding="utf-8") as f:
    cfg = json.load(f)

FILE_PATH        = cfg["source"]["file_path"]
SHEET_NAME       = cfg["source"]["sheet_name"]
CLEANED_OUT      = cfg["output"]["cleaned_file"]
PAGE_DATE_FIXES  = cfg["cleaning"]["page_date_corrections"]
SPANISH_MAP      = cfg["cleaning"]["spanish_day_map"]
DAY_TYPO_MAP     = cfg["cleaning"]["day_typo_map"]
TIME_COLS        = cfg["cleaning"]["time_columns"]
VALID_DAYS       = set(cfg["screening"]["valid_days"])
VALID_YEARS      = set(cfg["screening"]["valid_years"])
BU_CORRECT       = cfg["cleaning"]["business_unit_correct"]
BU_VARIANTS      = set(cfg["cleaning"]["business_unit_variants"])
PROJ_CORRECT     = cfg["cleaning"]["project_correct"]
PROJ_VARIANTS    = set(cfg["cleaning"]["project_variants"])
FILL_COLS        = cfg["cleaning"]["fill_from_previous_page_columns"]
COLUMN_ORDER     = cfg["cleaning"]["column_order"]

df = pd.read_excel(FILE_PATH, sheet_name=SHEET_NAME, dtype=str)
df.columns = df.columns.str.strip()
for col in df.columns:
    df[col] = df[col].str.strip()

print(f"Config loaded from secai_config.json")
print(f"Loaded {len(df):,} rows x {len(df.columns)} columns")

## Fix 0a · Date audit — one row per page

In [ ]:
def audit_date(v):
    if pd.isna(v) or str(v).strip() == "": return "⚠ blank"
    s = str(v).strip().replace(".", "/").replace("-", "/")
    try:
        d = pd.to_datetime(s, dayfirst=False)
        if d.year % 100 not in {yr % 100 for yr in VALID_YEARS}:
            return f"⚠ wrong year ({d.year})"
        return f"OK — {d.strftime('%b %d, %Y')} ({d.day_name()})"
    except Exception:
        return "⚠ unparseable"

page_summary = (
    df[["Page", "Date", "Day"]]
    .drop_duplicates(subset=["Page"])
    .assign(_pg=lambda d: pd.to_numeric(d["Page"], errors="coerce"))
    .sort_values("_pg").drop(columns="_pg")
    .reset_index(drop=True)
)
page_summary["Audit"] = page_summary["Date"].apply(audit_date)

try:
    import jinja2
    display(page_summary.style.apply(
        lambda r: ["background-color: #fdd" if "⚠" in str(r["Audit"]) else "" for _ in r],
        axis=1
    ))
except ImportError:
    display(page_summary)

## Fix 0b · Correct bad dates using page sequence context
Corrections are defined in `config/secai_config.json` → `cleaning.page_date_corrections`.

In [ ]:
# Each entry documents one date correction — field, raw value, fix, and reasoning
CORRECTIONS_LOG = [
    {
        "fix_id":   "F0b-01",
        "page":     "4",
        "field":    "Date",
        "raw":      "11-04-25",
        "fix":      "12/04/25",
        "problem":  "Month digit reads '11' but surrounding pages are all December. Likely OCR misread of '12'.",
        "evidence": "p3 = 12/03/25, p5 = 12/05/25. Sequence implies 12/04/25.",
    },
    {
        "fix_id":   "F0b-02",
        "page":     "7",
        "field":    "Date",
        "raw":      "10/08/05",
        "fix":      "12/08/25",
        "problem":  "Month reads '10' and year reads '05'. Both are OCR errors.",
        "evidence": "p6 = 12/06/25, p8 = 12/09/25. Day = Monday; 12/08/25 is a Monday.",
    },
    {
        "fix_id":   "F0b-03",
        "page":     "9",
        "field":    "Date",
        "raw":      "12/10/2015",
        "fix":      "12/10/25",
        "problem":  "Year recorded as 2015 — extra '1' inserted, likely OCR artifact.",
        "evidence": "p8 = 12/09/25, p10 = 12/11/25.",
    },
    {
        "fix_id":   "F0b-04",
        "page":     "10",
        "field":    "Date",
        "raw":      "12/11/23",
        "fix":      "12/11/25",
        "problem":  "Year reads '23' instead of '25'. Single digit transposition.",
        "evidence": "p9 = 12/10/25, p11 = 12/12/25. Day = Thursday; 12/11/25 is a Thursday.",
    },
    {
        "fix_id":   "F0b-05",
        "page":     "11",
        "field":    "Date",
        "raw":      "4/10/05",
        "fix":      "12/12/25",
        "problem":  "Date is fully garbled — entire value is an OCR misread.",
        "evidence": "p10 = 12/11/25, p12 = 12/13/25. Day = Friday; 12/12/25 is a Friday.",
    },
    {
        "fix_id":   "F0b-06",
        "page":     "12",
        "field":    "Date",
        "raw":      "12/12/85",
        "fix":      "12/13/25",
        "problem":  "Year reads '85' and day digit '3' was dropped.",
        "evidence": "p11 = 12/12/25 (Friday). Next day is Saturday 12/13/25. Day field blank.",
    },
    {
        "fix_id":   "F0b-07",
        "page":     "16",
        "field":    "Date",
        "raw":      "14/19/2025",
        "fix":      "12/18/25",
        "problem":  "Impossible date — month 14 does not exist. OCR merged adjacent characters.",
        "evidence": "p15 = 12/17/25, p18 = 12/19/25. Day = JUEVES (Thu); 12/18/25 is Thursday.",
    },
    {
        "fix_id":   "F0b-08",
        "page":     "18",
        "field":    "Date",
        "raw":      "12/19/20",
        "fix":      "12/19/25",
        "problem":  "Year reads '20' instead of '25'. Digit '5' misread as '0'.",
        "evidence": "p17 blank, p20 = 12/20/25. Day = Friday; 12/19/25 is a Friday.",
    },
    {
        "fix_id":   "F0b-09",
        "page":     "22",
        "field":    "Date",
        "raw":      "12/23/24",
        "fix":      "12/22/25",
        "problem":  "Year reads '24' and day reads '23' — p23 is also 12/23/25 so this must be the day before.",
        "evidence": "p23 = 12/23/25 (Tuesday). Day = Monday; 12/22/25 is a Monday.",
    },
]

total_fixed = 0
for entry in CORRECTIONS_LOG:
    mask = df["Page"] == entry["page"]
    n = int(mask.sum())
    df.loc[mask, entry["field"]] = PAGE_DATE_FIXES[entry["page"]]
    total_fixed += n
    parsed = pd.to_datetime(PAGE_DATE_FIXES[entry["page"]])
    print(f"  [{entry['fix_id']}] Page {entry['page']:>2s}: '{entry['raw']}'  →  '{PAGE_DATE_FIXES[entry['page']]}'  ({parsed.day_name()})  [{n} rows]")

print(f"\nTotal rows updated: {total_fixed}")

page_check = (
    df[["Page", "Date"]]
    .drop_duplicates(subset=["Page"])
    .assign(_pg=lambda d: pd.to_numeric(d["Page"], errors="coerce"))
    .sort_values("_pg").drop(columns="_pg")
    .reset_index(drop=True)
)
page_check["Audit"] = page_check["Date"].apply(audit_date)
remaining = page_check[page_check["Audit"].str.startswith("⚠")]
print("\nDate audit after corrections:")
if remaining.empty:
    print("  All dates clean.")
else:
    display(remaining)

## Fix 1 · Translate Spanish day names to English
Map defined in `config/secai_config.json` → `cleaning.spanish_day_map`.

In [ ]:
before = df["Day"].value_counts(dropna=False).to_dict()

df["Day"] = df["Day"].apply(
    lambda v: SPANISH_MAP.get(str(v).upper().strip(), v) if pd.notna(v) else v
)

changed = {k: v for k, v in before.items() if str(k).upper().strip() in SPANISH_MAP}
print(f"Translated {sum(changed.values())} rows across {len(changed)} Spanish value(s):")
for spanish, count in changed.items():
    print(f"  '{spanish}' ({count} rows)  →  '{SPANISH_MAP[str(spanish).upper().strip()]}'")

print("\nDistinct Day values after fix:")
display(df["Day"].value_counts(dropna=False).rename_axis("Day").reset_index(name="Count"))

## Fix 1 · Check — rows where Day is still not a standard weekday

In [ ]:
mask = ~df["Day"].apply(lambda v: str(v).strip().title() in VALID_DAYS if pd.notna(v) else False)
odd_days = df[mask][["Source File", "Page", "Date", "Day", "Employee Name"]].reset_index(drop=True)

print(f"{len(odd_days)} rows with non-standard Day values. Showing up to 10:")
display(odd_days.head(10))

## Fix 2 · Correct day typos and cross-check Date vs Day
Typo map defined in `config/secai_config.json` → `cleaning.day_typo_map`.

In [ ]:
before = df["Day"].value_counts(dropna=False).to_dict()

df["Day"] = df["Day"].apply(
    lambda v: DAY_TYPO_MAP.get(str(v).strip(), v) if pd.notna(v) else v
)

fixed = {k: v for k, v in before.items() if str(k).strip() in DAY_TYPO_MAP}
print("Typo corrections applied:")
for typo, count in fixed.items():
    print(f"  '{typo}' ({count} rows)  →  '{DAY_TYPO_MAP[str(typo).strip()]}'")

def safe_parse(v):
    if pd.isna(v) or str(v).strip() == "": return pd.NaT
    s = str(v).strip().replace(".", "/").replace("-", "/")
    try:
        return pd.to_datetime(s, dayfirst=False)
    except Exception:
        return pd.NaT

df["_parsed_date"]  = df["Date"].apply(safe_parse)
df["_expected_day"] = df["_parsed_date"].apply(lambda d: d.day_name() if pd.notna(d) else None)

comparable = df[df["Day"].notna() & df["Day"].ne("") & df["_expected_day"].notna()].copy()
conflicts = comparable[
    comparable["Day"].str.strip().str.title() != comparable["_expected_day"]
][["Source File", "Page", "Date", "_parsed_date", "_expected_day", "Day", "Employee Name"]]
conflicts = conflicts.rename(columns={"_parsed_date": "Date (parsed)", "_expected_day": "Expected Day"})

print(f"\nDate vs Day conflicts: {len(conflicts.reset_index(drop=True))} rows")
display(conflicts.reset_index(drop=True))

## Fix 3 · Normalise time columns to H:MMam/pm format
Columns defined in `config/secai_config.json` → `cleaning.time_columns`.

In [ ]:
def normalise_time(v):
    if pd.isna(v) or str(v).strip() == "": return v
    s = str(v).strip()

    # OCR fix: 'Pu' is a misread of 'PM' (bottom half of M looks like u)
    s = re.sub(r'\bPu\b', 'PM', s, flags=re.IGNORECASE)

    # Noon shorthand: '12m' or '12M' → '12:00PM'
    s = re.sub(r'^(\d{1,2})\s*[mM]$', r'\g<1>:00PM', s)

    # Duplicate suffix: '7 AM AM' → '7 AM'
    s = re.sub(r'\b(AM|PM)(\s+\1)+\b', r'\1', s, flags=re.IGNORECASE)

    m = re.match(r'^(\d{1,2})(?::(\d{1,2}))?\s*(AM|PM)?$', s.strip(), re.IGNORECASE)
    if not m: return v

    hour    = int(m.group(1))
    minutes = m.group(2)
    suffix  = m.group(3)

    if minutes is None:        minutes = "00"
    elif len(minutes) == 1:    minutes = minutes + "0"

    if suffix is None:
        if hour == 12:              suffix = "PM"
        elif hour in (0, 1, 2, 3): suffix = "AM"
        else:                       suffix = "AM"

    return f"{hour}:{minutes}{suffix.upper()}"

for col in TIME_COLS:
    before  = df[col].copy()
    df[col] = df[col].apply(normalise_time)
    changed = (before.fillna("") != df[col].fillna("")).sum()
    print(f"  {col:<14} — {changed} cells updated")

print("\nDistinct values after normalisation:")
for col in TIME_COLS:
    print(f"\n  {col}: {df[col].value_counts(dropna=False).to_dict()}")

## Fix 4 · Reorder columns and add Recorded Hours
Column order defined in `config/secai_config.json` → `cleaning.column_order`.

In [ ]:
df["Recorded Hours"] = df["Actual Hours"]

base_cols = [c for c in df.columns if not c.startswith("_")]
missing_from_order = [c for c in base_cols if c not in COLUMN_ORDER]
if missing_from_order:
    print(f"⚠ Columns not in config order (appended at end): {missing_from_order}")

final_order = [c for c in COLUMN_ORDER if c in df.columns] + missing_from_order
df = df[final_order]

print("Column order after fix:")
for i, col in enumerate(df.columns, 1):
    marker = "← new" if col == "Recorded Hours" else "← moved" if col in ("Title", "EIN") else ""
    print(f"  {i:>2}. {col}  {marker}")

## Fix 5 · Standardise Date format to M/D/YY

In [ ]:
def format_date(v):
    if pd.isna(v) or str(v).strip() == "": return v
    s = str(v).strip().replace(".", "/").replace("-", "/")
    try:
        d = pd.to_datetime(s, dayfirst=False)
        return f"{d.month}/{d.day}/{str(d.year)[-2:]}"
    except Exception:
        return v

before    = df["Date"].copy()
df["Date"] = df["Date"].apply(format_date)
changed   = (before.fillna("") != df["Date"].fillna("")).sum()
print(f"Date format standardised: {changed} cells updated")
display(
    df[["Page", "Date"]].drop_duplicates(subset=["Page"])
    .assign(_pg=lambda d: pd.to_numeric(d["Page"], errors="coerce"))
    .sort_values("_pg").drop(columns="_pg")
    .reset_index(drop=True)
)

## Fix 6 · Correct Business Unit / Project OCR variants and fill missing page fields
Variants and correct values defined in `config/secai_config.json` → `cleaning.*`.

In [ ]:
# Business Unit
bu_before   = df["Business Unit"].copy()
df["Business Unit"] = df["Business Unit"].apply(
    lambda v: BU_CORRECT if str(v).strip() in BU_VARIANTS else v
)
print(f"Business Unit: {(bu_before.fillna('') != df['Business Unit'].fillna('')).sum()} cells corrected to '{BU_CORRECT}'")

# Project
proj_before = df["Project"].copy()
df["Project"] = df["Project"].apply(
    lambda v: PROJ_CORRECT if str(v).strip() in PROJ_VARIANTS else v
)
print(f"Project:       {(proj_before.fillna('') != df['Project'].fillna('')).sum()} cells corrected to '{PROJ_CORRECT}'")

# Fill missing header fields from previous page
page_vals = (
    df[FILL_COLS + ["Page"]].dropna(subset=FILL_COLS, how="all")
    .drop_duplicates(subset=["Page"]).set_index("Page")
)
sorted_pages = sorted(df["Page"].dropna().unique(), key=lambda x: int(x) if str(x).isdigit() else 999)

fill_count = 0
for i, page in enumerate(sorted_pages):
    mask = df["Page"] == page
    for col in FILL_COLS:
        if df.loc[mask, col].isna().all() or df.loc[mask, col].eq("").all():
            for prev_page in reversed(sorted_pages[:i]):
                if prev_page in page_vals.index and pd.notna(page_vals.at[prev_page, col]):
                    fill_val = page_vals.at[prev_page, col]
                    df.loc[mask, col] = fill_val
                    fill_count += int(mask.sum())
                    print(f"  Page {page} · {col}: filled from page {prev_page} → '{fill_val}'")
                    break

print(f"\nTotal cells filled from previous page: {fill_count}")

---
## Export · Generate cleaned Excel file

In [ ]:
from openpyxl import load_workbook
from openpyxl.styles import PatternFill, Font
from openpyxl.utils import get_column_letter
from openpyxl.comments import Comment

df_raw = pd.read_excel(FILE_PATH, sheet_name=SHEET_NAME, dtype=str)
df_raw.columns = df_raw.columns.str.strip()
for col in df_raw.columns:
    df_raw[col] = df_raw[col].str.strip()

export_cols = [c for c in df.columns if not c.startswith("_")]
df_export   = df[export_cols].copy()

df_export.to_excel(CLEANED_OUT, index=False, sheet_name="Cleaned")

wb = load_workbook(CLEANED_OUT)
ws = wb["Cleaned"]
col_idx = {cell.value: cell.column for cell in ws[1]}

YELLOW = PatternFill("solid", fgColor="FFF3CD")
BOLD   = Font(bold=True)

changed_count = 0
for col in export_cols:
    if col not in df_raw.columns or col not in col_idx:
        continue
    for i in df_export.index:
        old = "" if pd.isna(df_raw.at[i, col]) else str(df_raw.at[i, col]).strip()
        new = "" if pd.isna(df_export.at[i, col]) else str(df_export.at[i, col]).strip()
        if old != new:
            cell = ws.cell(row=i + 2, column=col_idx[col])
            cell.fill = YELLOW
            c = Comment(f"Original: {old}", "SECAI")
            c.width, c.height = 180, 40
            cell.comment = c
            changed_count += 1

for cell in ws[1]:
    cell.font = BOLD
ws.freeze_panes = "A2"
for col_cells in ws.columns:
    width = max((len(str(c.value)) if c.value else 0) for c in col_cells)
    ws.column_dimensions[get_column_letter(col_cells[0].column)].width = min(width + 2, 30)

wb.save(CLEANED_OUT)
print(f"Saved → {CLEANED_OUT}")
print(f"  Rows          : {len(df_export):,}")
print(f"  Columns       : {len(export_cols)}")
print(f"  Changed cells : {changed_count}")